<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Electricity load for many customers

**[Electricity load for many customers](https://learning.nextia-ai.com/courses/forecasting/m09/electricity-load-for-many-customers/)** · Forecasting and Time Series: Predict What Comes Next · Case study 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** forecast the hourly electricity use of 80 real customers for the next day, compare a per-series model (MSTL), one global LightGBM model and a pretrained model (Chronos-Bolt) in the course's backtest, score their 90% quantiles with the pinball loss and the coverage, and price the energy that a supplier buys the day before.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page; the notebook runs in about 4 minutes after the installs |
| **Needs** | An internet connection for the setup cells and the model download (about 190 MB, once). No account, no key and no GPU. |
| **You should know** | The backtest, from [Test like the future](https://learning.nextia-ai.com/courses/forecasting/m03/test-like-the-future-backtesting/). MSTL, from [Several seasons and long series](https://learning.nextia-ai.com/courses/forecasting/m04/several-seasons-and-long-series/). LightGBM and Chronos-Bolt, from [Gradient boosting with LightGBM](https://learning.nextia-ai.com/courses/forecasting/m05/gradient-boosting-with-lightgbm/) and [Deep and foundation models](https://learning.nextia-ai.com/courses/forecasting/m05/deep-and-foundation-models/). The pinball loss, from [Score the range](https://learning.nextia-ai.com/courses/forecasting/m07/score-the-range/). |
| **Data** | UCI ElectricityLoadDiagrams20112014, CC BY 4.0: 80 real customers in Portugal, kWh per hour, 2013 and 2014 |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, LightGBM 4.7.0, chronos-forecasting 2.3.2 and PyTorch 2.15.0 (CPU), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m09/electricity-load-for-many-customers/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M09-L02-electricity-load-for-many-customers/electricity-load-for-many-customers-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, lightgbm 4.7.0, scikit-learn 1.9.1, chronos-forecasting 2.3.2, torch 2.15.0, and matplotlib. It needs an internet connection. PyTorch is large (about 200 MB to download), so the first run of this cell can take 2 to 3 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

The cell reads the versions without loading the packages. This notebook never loads PyTorch and LightGBM in the same process: on macOS, that combination crashed. Chronos-Bolt runs in its own process in section 4.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 mlforecast==1.1.0 lightgbm==4.7.0 scikit-learn==1.9.1 chronos-forecasting==2.3.2 torch==2.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ['pandas', 'statsforecast', 'mlforecast', 'lightgbm', 'scikit-learn', 'chronos-forecasting', 'torch']:
    print(_name, _md.version(_name))

## Setup: get the data

Run the next cell. It downloads two files into a new folder `electricity-forecasting` and checks each file's checksum:

- `electricity_hourly_80.csv.gz` (3.7 MB): the hourly electricity use of 80 real customers in Portugal, 2013 and 2014, from the UCI Machine Learning Repository;
- `backtest.py`: the course's backtest helper, the same file as in every lesson of the course.

It needs an internet connection and no account, and takes a few seconds. You should see `Ready: electricity_hourly_80.csv.gz (3.7 MB) and backtest.py.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `electricity-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

**Source and licence.** "ElectricityLoadDiagrams20112014" by Artur Trindade, UCI Machine Learning Repository, <https://doi.org/10.24432/C58C86>, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Nextia Learning kept 80 of the 370 customers and the years 2013 and 2014, and added the 15-minute values up to kWh per hour ([dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M09-L02-electricity-load-for-many-customers/dataset.md)).

In [ ]:
# Setup: download the data and the backtest helper into electricity-forecasting/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

FILES = {
    "electricity_hourly_80.csv.gz": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/M09-L02-electricity-load-for-many-customers/electricity_hourly_80.csv.gz",
        "ef825f470a6f39cc36055aedff6bcaa35ae95a9cfaf9a7ad5f720c29e476510c"
    ],
    "backtest.py": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/backtest.py",
        "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a"
    ]
}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("electricity-forecasting").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, (url, expected) in FILES.items():
    path = PROJECT / name
    if path.exists() and sha256(path) == expected:
        continue
    if (HERE / name).exists():  # a copy next to the notebook
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder electricity-forecasting and run again.")

os.chdir(PROJECT)
print(f"Ready: electricity_hourly_80.csv.gz ({Path('electricity_hourly_80.csv.gz').stat().st_size / 1e6:.1f} MB) and backtest.py.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The hours

One row is one hour. Each of the 80 columns is one customer, with the electricity it used in that hour, in **kWh** (kilowatt-hours). The next cell loads the file, puts it into the long format of the backtest (`unique_id`, `ds`, `y`), and compares the customers' sizes.

In [ ]:
wide = pd.read_csv("electricity_hourly_80.csv.gz", index_col="hour", parse_dates=True)
size = wide.mean()  # each customer's mean kWh per hour
print(wide.shape, wide.index.min(), "to", wide.index.max())
print(f"zero hours: {(wide == 0).sum().sum():,} of {wide.size:,}")
print("mean kWh per hour: smallest", round(size.min(), 1), "| median", round(size.median(), 1), "| largest", round(size.max(), 1))

> **Check.** You should see `(17520, 80)`, from 2013-01-01 00:00 to 2014-12-31 23:00, 51 zero hours, and customers from 5.2 to 6311.4 kWh per hour (median 130.6). The largest customer uses about 1,200 times as much as the smallest.

> **Your turn.** What share of all the kWh do the **5 largest** customers use? Use `size`, and put the share in `top5_share`, rounded to 3 decimals. Then run the check cell.

In [ ]:
top5_share = ...  # the 5 largest customers' share of all kWh

In [ ]:
expect('top5_share', top5_share, 0.457, tolerance=0.0005)

## 2. The clock changes

Portugal moves its clocks one hour forward on the last Sunday of March and one hour back on the last Sunday of October. The source says what happens in this data: in March, the hour from 01:00 to 02:00 is (mostly) zero; in October, that hour holds **two** hours of use. The next cell shows the total of all customers at 00:00, 01:00 and 02:00 on those four days.

In [ ]:
total_raw = wide.sum(axis=1)
def clock_change_days(years):
    """The last Sunday of March and of October, every year."""
    for year in years:
        for month in (3, 10):
            last = pd.Timestamp(year, month, 1) + pd.offsets.MonthEnd(0)
            yield month, last - pd.Timedelta(days=(last.dayofweek + 1) % 7)

for month, day in clock_change_days([2013, 2014]):
    print(day.date(), total_raw.loc[day:day + pd.Timedelta(hours=2)].round().astype(int).tolist())

> **Check.** You should see four lines. In March, the middle value is about a third of its neighbours (4880 and 3757). In October, it is almost double the next hour (26885 and 28047).

We keep a simple 24-hour day and fix the two hours: in March, the mean of 00:00 and 02:00; in October, half of the value.

In [ ]:
def fix_clock_changes(wide):
    """Spring: the hour 01:00 holds only part of an hour: use the mean of 00:00 and 02:00.
    Autumn: the hour 01:00 holds two hours: halve it."""
    w = wide.copy()
    for month, day in clock_change_days(sorted(set(w.index.year))):
        h0, h1, h2 = (day + pd.Timedelta(hours=k) for k in range(3))
        w.loc[h1] = (w.loc[h0] + w.loc[h2]) / 2 if month == 3 else w.loc[h1] / 2
    return w

wide = fix_clock_changes(wide)
load = wide.reset_index().melt(id_vars="hour", var_name="unique_id", value_name="y").rename(columns={"hour": "ds"})
total = wide.sum(axis=1)
print(len(load), "rows |", total.loc["2014-10-26 00:00":"2014-10-26 02:00"].round().astype(int).tolist())

> **Check.** You should see `1401600 rows | [22677, 14024, 15483]`: the October hour is now close to its neighbours.

## 3. The seasons

Electricity use has a **daily** season (night, morning, evening) and a **weekly** season (offices close at the weekend). The autocorrelation tells how much the total at one hour looks like the total 1, 24 and 168 hours before. The ratio of a customer's weekday mean to its weekend mean shows its weekly season.

> **Predict.** How many of the 80 customers use more than 1.2 times as much on a weekday as at the weekend? Then run the cell.

In [ ]:
print("autocorrelation of the total, lag 1, 24, 168 hours:", {lag: round(float(total.autocorr(lag)), 3) for lag in (1, 24, 168)})
week_ratio = wide[wide.index.dayofweek < 5].mean() / wide[wide.index.dayofweek >= 5].mean()
print("customers with weekday / weekend above 1.2:", int((week_ratio > 1.2).sum()), "| largest:", week_ratio.idxmax(), round(week_ratio.max(), 2))

> **Check.** You should see 0.954, 0.982 and 0.97: the same hour yesterday and the same hour last week are both very close. Only 8 customers use more than 1.2 times as much on weekdays; the strongest is MT_368 (2.61). For most customers, the daily season is much stronger than the weekly one.

## 4. The backtest

**The decision.** The supplier buys each hour of tomorrow's energy today, before the market closes at noon. Our forecast is made at **11:00**. It covers 36 hours, from 12:00 today (step 1) to 23:00 tomorrow (step 36), and the energy is bought for tomorrow's 24 hours only: steps 13 to 36.

**The costs (assumptions).** A kWh short is bought at the last minute and costs **€0.09** more. A kWh bought and not used is sold back at a loss of **€0.01**. The best amount to buy is the quantile at the critical ratio 0.09 / (0.09 + 0.01) = **0.9**: the **90% quantile**, q90.

**The backtest.** 12 origins, 26 days apart (so the weekday changes), from 10 February to 23 November 2014, at 11:00. December is kept for the final test.

In [ ]:
H, FIRST_STEP = 36, 13          # 12:00 today is step 1, 23:00 tomorrow step 36; tomorrow 00:00 is step 13
ORIGINS = bt.origins("2014-11-23 11:00", n=12, every="26D")
FINAL = bt.origins("2014-12-29 11:00", n=6, every="5D")
WINDOW = pd.Timedelta(weeks=4)  # MSTL and the naive method see the last 4 weeks
SHORT, OVER = 0.09, 0.01        # euros per kWh (assumptions)
RATIO = SHORT / (SHORT + OVER)

def capacity_cost(short=SHORT, over=OVER):
    """The cost of the energy bought the day before, per day: each kWh short costs `short` euros, each kWh
    bought and not used `over` euros. The column is the amount bought per customer and hour."""
    def cost(fc, column="q90"):
        buy = fc[column].clip(lower=0)
        lack, extra = np.maximum(fc["y"] - buy, 0).sum(), np.maximum(buy - fc["y"], 0).sum()
        return float((lack * short + extra * over) / fc["origin"].nunique())
    return cost

def tomorrow(fc):
    """Keep tomorrow's 24 hours: the hours that the energy is bought for."""
    out = fc[fc["step"] >= FIRST_STEP].copy()
    out.attrs = fc.attrs
    return out

cap = capacity_cost()
print("critical ratio", RATIO, "| origins", ORIGINS[0], "to", ORIGINS[-1], "| final", FINAL[0], "to", FINAL[-1])

> **Check.** You should see `critical ratio 0.9`, origins from 2014-02-10 11:00 to 2014-11-23 11:00, and final origins from 2014-12-04 to 2014-12-29.

Now the four methods, each as a forecaster for `bt.backtest()` with a `q10` and a `q90` column:

- **same hour last week** (the baseline) and **MSTL** with a daily (24) and a weekly (168) season: statsforecast, one model **per customer**, fitted on the last 4 weeks, with their 80% range (q10 to q90);
- **LightGBM, global**: **one** model for all 80 customers, plus two quantile models for q10 and q90. Each customer is first scaled to its own mean and spread. Only lags of 48 hours or more: at 11:00, tomorrow's 23:00 is 36 hours away. It is fitted on 26 weeks, at every fourth origin;
- **Chronos-Bolt small**, a pretrained model with no training, in its own process.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import MSTL, SeasonalNaive

def stats(model, name):
    """A statsforecast model with an 80% range, fitted on the last 4 weeks before the origin."""
    def forecaster(train, h, future):
        recent = train[train["ds"] > train["ds"].max() - WINDOW][["unique_id", "ds", "y"]]
        f = StatsForecast(models=[model], freq="h", n_jobs=1).forecast(df=recent, h=h, level=[80])
        return pd.DataFrame({"unique_id": f["unique_id"], "ds": f["ds"], "yhat": f[name].clip(lower=0),
                             "q10": f[f"{name}-lo-80"].clip(lower=0), "q90": f[f"{name}-hi-80"].clip(lower=0)})
    return forecaster

In [ ]:
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from mlforecast.target_transforms import LocalStandardScaler

class GlobalLightGBM:
    """One LightGBM model for all customers, and two more for q10 and q90. Fitted on the last 26 weeks."""
    def __init__(self, weeks=26):
        self.weeks = weeks

    def fit(self, train):
        p = dict(n_estimators=300, learning_rate=0.05, num_leaves=63, verbose=-1, n_jobs=1, random_state=35)
        models = {"yhat": lgb.LGBMRegressor(**p), "q10": lgb.LGBMRegressor(objective="quantile", alpha=0.1, **p),
                  "q90": lgb.LGBMRegressor(objective="quantile", alpha=0.9, **p)}
        self.model = MLForecast(models=models, freq="h", lags=[48, 72, 168, 336],
                                lag_transforms={48: [RollingMean(24)], 168: [RollingMean(168)]},
                                date_features=["hour", "dayofweek"], target_transforms=[LocalStandardScaler()])
        recent = train[train["ds"] > train["ds"].max() - pd.Timedelta(weeks=self.weeks)][["unique_id", "ds", "y"]]
        self.model.fit(recent)

    def predict(self, train, h, future):
        recent = train[train["ds"] > train["ds"].max() - WINDOW][["unique_id", "ds", "y"]]
        f = self.model.predict(h, new_df=recent)
        return f[["unique_id", "ds", "yhat", "q10", "q90"]].assign(**{c: f[c].clip(lower=0) for c in ("yhat", "q10", "q90")})

The next cell writes the Chronos part into its own file, `chronos_forecasts.py`. The function after it runs that file in a new Python process, for all origins at once, and reads its forecasts back. The model is downloaded once (about 190 MB) at a fixed revision.

In [ ]:
%%writefile chronos_forecasts.py
"""Chronos-Bolt small forecasts for the electricity case study, in their own process.

On macOS, PyTorch and LightGBM in one process crashed (the forecasting course's brief), so the notebook runs
this file with `python chronos_forecasts.py <load.pkl> <origins.json> <h> <out.pkl>`. It reads the cleaned
hourly load (unique_id, ds, y), forecasts h hours after every origin from the last 2,048 hours (the model's
longest context), and writes q10, q50 and q90 per customer and hour, and the seconds it took.
"""

import json
import sys
import time

import pandas as pd
import torch
from chronos import BaseChronosPipeline

REVISION = "772f3d25d38aec6d914c8949dab4462e2d46f5d8"
LEVELS = [0.1, 0.5, 0.9]

load_path, origins_path, h, out_path = sys.argv[1], sys.argv[2], int(sys.argv[3]), sys.argv[4]
load = pd.read_pickle(load_path).sort_values(["unique_id", "ds"])
origins = [pd.Timestamp(o) for o in json.loads(open(origins_path).read())]
start = time.perf_counter()
pipe = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-small", revision=REVISION,
                                           device_map="cpu", torch_dtype=torch.float32)
load_seconds = time.perf_counter() - start
ids = sorted(load["unique_id"].unique())
out, seconds = [], 0.0
for origin in origins:
    train = load[load["ds"] <= origin]
    ctx = [torch.tensor(g["y"].to_numpy()[-2048:], dtype=torch.float32) for _, g in train.groupby("unique_id", sort=True)]
    t0 = time.perf_counter()
    q, _ = pipe.predict_quantiles(ctx, prediction_length=h, quantile_levels=LEVELS)
    seconds += time.perf_counter() - t0
    q = q.numpy().clip(0)
    hours = pd.date_range(origin + pd.Timedelta(hours=1), periods=h, freq="h")
    for k, uid in enumerate(ids):
        out.append(pd.DataFrame({"unique_id": uid, "ds": hours, "q10": q[k, :, 0], "q50": q[k, :, 1], "q90": q[k, :, 2]}))
result = pd.concat(out, ignore_index=True)
result.attrs["fit_seconds"] = seconds
result.attrs["load_seconds"] = load_seconds
result.to_pickle(out_path)
print(f"Chronos-Bolt small: {len(origins)} origins x {len(ids)} customers in {seconds:.1f} s (model loaded in {load_seconds:.1f} s)")

In [ ]:
import json, subprocess, sys

def run_chronos(df, origins, name):
    """Chronos-Bolt in its own process; returns a forecaster that reads its forecasts, and the seconds."""
    df[["unique_id", "ds", "y"]].to_pickle(f"{name}_load.pkl")
    Path(f"{name}_origins.json").write_text(json.dumps([str(o) for o in origins]))
    run = subprocess.run([sys.executable, "chronos_forecasts.py", f"{name}_load.pkl", f"{name}_origins.json",
                          str(H), f"{name}_chronos.pkl"], capture_output=True, text=True)
    if run.returncode:
        print(run.stderr[-2000:])
        raise SystemExit("Chronos did not run: read the error above.")
    print(run.stdout.strip())
    pre = pd.read_pickle(f"{name}_chronos.pkl")

    def forecaster(train, h, future):
        f = future[["unique_id", "ds"]].merge(pre, on=["unique_id", "ds"])
        return f.assign(yhat=f["q50"])[["unique_id", "ds", "yhat", "q10", "q90"]]
    return forecaster, pre.attrs.get("fit_seconds", 0.0)

One function runs the four methods at a list of origins and scores them on tomorrow's hours: MASE (scale: the same hour last week), WAPE and bias for the point forecast; for q90, the **pinball loss** at 0.9 (divided by the mean load) and the share of hours **below q90** (it should be 0.9); the share inside the 80% range (it should be 0.8); and the cost per day of buying the forecast and of buying q90.

> **Predict.** Which method has the lowest cost at q90? Will the one with the best MASE also have the lowest cost? Then run the cell (about 2 minutes: MSTL takes most of it).

In [ ]:
COLS = ["method", "MASE", "WAPE %", "bias %", "pinball q90", "below q90", "80% range holds", "fit s",
        "cost at forecast", "cost $"]

def row_for(name, fc, df):
    fc = tomorrow(fc)
    r = bt.score(fc, df, name, season=168, cost=cap, cost_column="q90")
    r["pinball q90"] = round(float(bt.pinball(fc["y"], fc["q90"], 0.9).mean() / fc["y"].mean()), 4)
    r["below q90"] = round(float((fc["y"] <= fc["q90"]).mean()), 3)
    r["80% range holds"] = bt.coverage(fc, "q10", "q90")
    r["cost at forecast"] = round(cap(fc, "yhat"))
    return r

def run_all(df, origins, tag):
    out = {"same hour last week": bt.backtest(df, stats(SeasonalNaive(season_length=168), "SeasonalNaive"), h=H, origins=origins),
           "MSTL (24, 168)": bt.backtest(df, stats(MSTL(season_length=[24, 168]), "MSTL"), h=H, origins=origins),
           "LightGBM, global": bt.backtest(df, GlobalLightGBM(), h=H, origins=origins, refit_every=4)}
    chronos, seconds = run_chronos(df, origins, tag)
    out["Chronos-Bolt small"] = bt.backtest(df, chronos, h=H, origins=origins)
    out["Chronos-Bolt small"].attrs["fit_seconds"] = seconds
    table = pd.DataFrame([row_for(name, fc, df) for name, fc in out.items()])[COLS]
    return out, table.rename(columns={"cost $": "cost at q90"})

fcs, table = run_all(load, ORIGINS, "selection")
print(table.astype(str).to_string(index=False))

> **Check.** You should see four rows. Chronos-Bolt has the best MASE (0.657), the lowest pinball loss (0.0142) and the lowest cost at q90 (915 euros a day), and 91.3% of the hours are below its q90. LightGBM is second (1,015 euros); its q90 is a little too low (0.885). MSTL's q90 is too high (0.933), and it took about 70 seconds. The fit times depend on your computer.

Every method's cost at q90 is far below its cost at the forecast. With these costs, buying the forecast is the expensive choice.

> **Your turn.** What does it cost per day to buy Chronos's **median** (`yhat`) instead of its q90? Use `cap()` on tomorrow's hours of `fcs["Chronos-Bolt small"]` with the column `"yhat"`, and put the rounded number in `median_cost`. Then run the check cell.

In [ ]:
median_cost = ...  # the cost per day of buying Chronos's median

In [ ]:
expect('median_cost', median_cost, 1704)

## 5. Per customer and by size

A global model shares what it learns between customers. Does that help the small ones? The next cell counts, per customer, which method has the lowest pinball loss at 0.9, and splits the cost into the 40 smallest and the 40 largest customers.

In [ ]:
pin = {}
for name, fc in fcs.items():
    f = tomorrow(fc)
    loss = bt.pinball(f["y"], f["q90"], 0.9)
    pin[name] = loss.groupby(f["unique_id"]).mean() / f.groupby("unique_id")["y"].mean()
pin = pd.DataFrame(pin)  # one row per customer, one column per method
print("lowest pinball at 0.9, customers per method:", pin.idxmin(axis=1).value_counts().to_dict())
small = size.nsmallest(40).index
for name, fc in fcs.items():
    f = tomorrow(fc)
    s, b = f[f["unique_id"].isin(small)], f[~f["unique_id"].isin(small)]
    print(f"{name:20} cost small {cap(s):6.0f}, large {cap(b):6.0f} | below q90: small {(s['y'] <= s['q90']).mean():.3f}, large {(b['y'] <= b['q90']).mean():.3f}")

> **Check.** You should see Chronos-Bolt best for 62 customers, LightGBM for 17 and MSTL for 1. The 40 large customers carry most of the cost. LightGBM's q90 is too low for the small customers (0.873 below q90).

## 6. One purchase for all customers

A real supplier does not buy for each customer alone. The market settles the **total**: one customer's surplus covers another's shortfall. Two ways to buy the total:

- **add up the 80 customers' q90**: easy, but each q90 has its own safety margin, and the margins add up;
- **forecast the total itself** and buy its q90.

> **Predict.** Is the sum of 80 q90 values about the q90 of the total, or much more? Then run the cell (about 30 seconds).

In [ ]:
tot = total.rename("y").reset_index().rename(columns={"hour": "ds"}).assign(unique_id="all customers")
chronos_total, _ = run_chronos(tot, ORIGINS, "total")
fc_total = tomorrow(bt.backtest(tot, chronos_total, h=H, origins=ORIGINS))
each = tomorrow(fcs["Chronos-Bolt small"])
summed = each.groupby(["ds", "origin"], as_index=False)[["y", "q90"]].sum()
print(f"each customer alone:   {cap(each):5.0f} euros a day")
print(f"sum of the 80 q90:     {cap(summed):5.0f} euros a day, below q90 {(summed['y'] <= summed['q90']).mean():.3f}")
print(f"the total's own q90:   {cap(fc_total):5.0f} euros a day, below q90 {(fc_total['y'] <= fc_total['q90']).mean():.3f}")

> **Check.** You should see 915, then 664, then 393 euros a day. The total's own q90 holds 93.4% of the hours; the sum of the 80 q90 values holds far more.

> **Your turn.** In what share of tomorrow's hours is the real total **below the sum** of the 80 q90 values? Use `summed`, and put the share in `sum_below`, rounded to 3 decimals. Then run the check cell.

In [ ]:
sum_below = ...  # the share of hours with the total below the sum of the 80 q90

In [ ]:
expect('sum_below', sum_below, 0.997, tolerance=0.0005)

A quantile of a sum is not the sum of the quantiles. The customers do not all have a high hour at the same time, so the total varies less, compared with its size, than each customer does.

## 7. The final test, once

The comparison chose Chronos-Bolt. Now the six December origins, which no step has seen, from 4 to 29 December 2014: the days 5, 10, 15, 20, 25 and 30 December. We run every method once and change nothing after (about 1 minute).

In [ ]:
fin, final_table = run_all(load, FINAL, "final")
print(final_table.astype(str).to_string(index=False))
print({str((o + pd.Timedelta(days=1)).date()): round(cap(g)) for o, g in tomorrow(fin["Chronos-Bolt small"]).groupby("origin")})

> **Check.** Chronos-Bolt has the lowest cost again (1,354 euros a day), but every method is worse than in the comparison: MASE above 1.3, and bias of +7.7% to +8.4%. The last line shows Chronos's cost per day. Christmas Day costs 3,454 euros: about three times a normal day.

On 25 December, many businesses close, and no method here knows the calendar. The lesson page discusses what to do.

## 8. New input: the purchase for 31 December

The last step runs the chosen pipeline at 11:00 on 30 December 2014 for the total, and gives the energy to buy for each hour of 31 December. The data has the real use of that day, so we can score it too.

In [ ]:
LAST = pd.Timestamp("2014-12-30 11:00")
chronos_new, _ = run_chronos(tot, [LAST], "new")
plan = tomorrow(bt.backtest(tot, chronos_new, h=H, origins=[LAST]))
print(f"buy {plan['q90'].sum() / 1000:.1f} MWh (median forecast {plan['yhat'].sum() / 1000:.1f}) | real {plan['y'].sum() / 1000:.1f} MWh "
      f"| hours short: {int((plan['y'] > plan['q90']).sum())} | cost {cap(plan):.0f} euros")
print(plan[["ds", "y", "yhat", "q90"]].iloc[[0, 8, 12, 19]].round(0).to_string(index=False))

> **Check.** You should see `buy 633.8 MWh`, a median forecast of 583.8 MWh, a real use of 538.5 MWh, no hour short, and a cost of 953 euros. New Year's Eve used less than the forecast, so most of the cost is energy sold back.

## 9. Failure case: the clock change, one week later

What if we had not fixed the clock changes? "Same hour last week" copies the hour 01:00 of 26 October 2014 into 2 November.

In [ ]:
raw_total = pd.read_csv("electricity_hourly_80.csv.gz", index_col="hour", parse_dates=True).sum(axis=1)
print("2 November 2014, 01:00, all customers:", round(total["2014-11-02 01:00"]),
      "| same hour last week without the fix:", round(raw_total["2014-10-26 01:00"]),
      "| with the fix:", round(total["2014-10-26 01:00"]))

> **Check.** You should see a real use of 15464 kWh, a forecast of 28047 kWh without the fix (81% too high) and 14024 with it. LightGBM's lag 168 and lag 336 copy the same error into its inputs.

## Next

You have taken 80 customers' meter data to a day-ahead purchase with its cost. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m09/electricity-load-for-many-customers/) has the charts, the recommendation, the limits, how to make this a portfolio piece, and the knowledge checks.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Retail demand with a hierarchy](https://learning.nextia-ai.com/courses/forecasting/m09/retail-demand-with-a-hierarchy/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m09/electricity-load-for-many-customers/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real and anonymous: UCI ElectricityLoadDiagrams20112014, CC BY 4.0.